# ML Stock Ranking Prototype

Goal: build a first supervised ranking/classification target using the existing feature table and cached price data.

Suggested next step after this notebook works well: move feature/label preparation into `ai_models/alpha_model.py`.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT

WindowsPath('c:/Users/bhuiyana/Documents/GitHub/ai-investment-lab')

In [ ]:
import pandas as pd

from ai_models.feature_builder import build_feature_table

feature_result = build_feature_table(
    fundamentals_path="data/fundamentals_cache.parquet",
    prices_path="data/prices_cache.parquet",
    treasury_path="data/treasury_yields_cache.parquet",
    benchmark_ticker="SPY",
)
features = feature_result.features.reset_index()
features.head()

In [ ]:
prices = pd.read_parquet(ROOT / "data" / "prices_cache.parquet")
prices["Ticker"] = prices["Ticker"].astype(str).str.upper().str.strip()
prices["Date"] = pd.to_datetime(prices["Date"], errors="coerce")
prices["AdjClose"] = pd.to_numeric(prices["AdjClose"], errors="coerce")
prices = prices.dropna(subset=["Ticker", "Date", "AdjClose"])
prices = prices.sort_values(["Ticker", "Date"])
prices.head()

In [ ]:
# Prototype label: future 63-business-day return by ticker.
future_returns = []
for ticker, g in prices.groupby("Ticker"):
    s = g.set_index("Date")["AdjClose"].sort_index()
    fwd_63 = s.shift(-63) / s - 1.0
    if not fwd_63.dropna().empty:
        future_returns.append({"Ticker": ticker, "ForwardReturn63D": float(fwd_63.dropna().iloc[-1])})

labels = pd.DataFrame(future_returns)
dataset = features.merge(labels, on="Ticker", how="inner")
dataset.head()

In [ ]:
# Optional if scikit-learn is installed.
try:
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import mean_absolute_error, r2_score

    numeric = dataset.select_dtypes(include=["number"]).drop(columns=["ForwardReturn63D"])
    y = dataset["ForwardReturn63D"]
    X = numeric.fillna(numeric.median(numeric_only=True))

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
    model = RandomForestRegressor(n_estimators=200, random_state=42)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print({"mae": mean_absolute_error(y_test, pred), "r2": r2_score(y_test, pred)})
except ModuleNotFoundError:
    print("Install scikit-learn to run the first ML ranking prototype.")